# آموزشِ صدا — نسخهٔ Kaggle

**فرقش با Colab:** اینجا اجرا روی سرورِ Kaggle در پس‌زمینه می‌رود.
مرورگر را می‌بندید، کامپیوتر را خاموش می‌کنید، اینترنتتان قطع می‌شود —
هیچ‌کدام اثری ندارد.

## کاری که باید بکنید (یک بار)

۱. در پنلِ راست: `Session options`
   - **Accelerator** ← `GPU T4 x2`
   - **Internet** ← `On`
     (اگر خاموش بود، حسابتان باید با شمارهٔ موبایل تأیید شده باشد:
     `Settings` ← `Phone Verification`)
۲. سلولِ «تنظیمات» را نگاه کنید
۳. دکمهٔ **`Save Version`** بالا سمتِ راست ←
   **`Save & Run All (Commit)`** ← `Save`

همین. حالا می‌توانید تب را ببندید.

## بعدش

یکی-دو ساعت بعد به همان صفحه برگردید و از تبِ `Output` دو فایل را
دانلود کنید. بعد در درایو، پوشهٔ `voice-models` بگذاریدشان.

---

منطقِ قدم‌ها در مخزن است و همین نوت‌بوک از آنجا می‌گیردش — همان کدی
که نسخهٔ Colab هم اجرا می‌کند.

## ۱ — کارتِ گرافیک و اینترنت

هر دو باید روشن باشند، وگرنه بقیه بی‌معنی است.

In [ ]:
import subprocess, sys, os, shutil, json, time

g = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True)
name = (g.stdout or b'').decode().strip()
if g.returncode != 0 or not name:
    raise SystemExit('GPU روشن نیست: Session options ← Accelerator ← GPU T4')
print('کارت:', name.splitlines()[0])

# اینترنت را همین‌جا می‌سنجیم، نه سه سلول بعد وقتی دانلود می‌شکند.
r = subprocess.run(['curl', '-sSfI', '-m', '20',
                    'https://raw.githubusercontent.com'],
                   capture_output=True)
if r.returncode:
    raise SystemExit('اینترنت خاموش است: Session options ← Internet ← On')
print('اینترنت: وصل')

## ۲ — تنظیمات

In [ ]:
VOICE = 'razavi'

DRIVE_IDS = [
    '1YRI2p7Qv3hh2dcNPMZDmbNUel0XCYWKX',
    '1cBUasKKB2Q5JjLfpZfyjBC7ZNo72KAiD',
    '1izlhA9PRU0VWcmL-Gw7lFaW2LJ3nLUKv',
    '1QdJzUi8sk5LhuUqjHeCi9Kb4UgRYq4P5',
]

SR      = '40k'
EPOCHS  = 150
BATCH   = 8
SAVE_EVERY = 25

# در Kaggle خروجی از این پوشه برداشته می‌شود.
OUT  = '/kaggle/working'
WORK = OUT + '/work'

## ۳ — کد

In [ ]:
os.chdir(OUT)
if not os.path.isdir(OUT + '/rvc'):
    subprocess.run(['git', 'clone', '--depth', '1', '-q',
                    'https://github.com/RVC-Project/'
                    'Retrieval-based-Voice-Conversion-WebUI',
                    OUT + '/rvc'], check=True)

RAW = 'https://raw.githubusercontent.com/mahdighandi1989/Content-Engine/main/tools'
for f in ('rvcpipe.py', 'dsprep.py'):
    subprocess.run(['curl', '-sSLf', RAW + '/' + f,
                    '-o', OUT + '/' + f], check=True)
sys.path.insert(0, OUT)
import rvcpipe as P, dsprep as D

ROOT = OUT + '/rvc'
os.makedirs(WORK, exist_ok=True)
print('قدم‌ها:', [n for n, _ in P.steps(VOICE, '/x', ROOT, sr=SR)])

## ۴ — وابستگی‌ها

In [ ]:
deps = P.TRAIN_DEPS + D.DS_DEPS + ['gdown']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q']
               + deps, check=True)
print('نصب شد:', len(deps), 'بسته')

## ۵ — وزن‌های پایه

حدود ۵۵۰ مگابایت. پروانه‌ها سنجیده شده: کدِ RVC ‏MIT · وزن‌ها ‏MIT ·
ContentVec ‏MIT · RMVPE ‏Apache-2.0 · مدلِ گوینده ‏Apache-2.0.

In [ ]:
os.chdir(ROOT)
hf = shutil.which('hf') or shutil.which('huggingface-cli')
for cmd in P.assetCmds_(py=sys.executable, sr=SR, hf=hf or 'hf'):
    if cmd[:3] == [sys.executable, '-m', 'pip']:
        continue
    if cmd[0] in ('hf', 'huggingface-cli'):
        cmd[0] = hf or cmd[0]
    r = subprocess.run(cmd)
    if r.returncode:
        raise SystemExit('دانلود ناموفق: ' + ' '.join(cmd[:4]))
print('وزن‌ها آماده‌اند')

## ۶ — گرفتنِ ضبط‌ها

In [ ]:
import gdown
RAWDIR = WORK + '/raw'
os.makedirs(RAWDIR, exist_ok=True)
srcs = []
for i, fid in enumerate(DRIVE_IDS):
    dst = os.path.join(RAWDIR, 'in%d' % (i + 1))
    if not (os.path.exists(dst) and os.path.getsize(dst) > 100000):
        gdown.download(id=fid, output=dst, quiet=True)
    if os.path.exists(dst) and os.path.getsize(dst) > 100000:
        srcs.append(dst)
    else:
        print('نیامد (دسترسی؟):', fid)
print('%d از %d ضبط آماده است' % (len(srcs), len(DRIVE_IDS)))
if not srcs:
    raise SystemExit('هیچ فایلی نیامد — اشتراکِ فایل‌ها باید «هر کسی با لینک» باشد')

## ۷ — جداکردنِ موسیقی و ساختِ دیتاست

سه دروازه: فاصله‌های بلند، کفِ هر تکه، و شباهت به گویندهٔ غالب.

In [ ]:
DS = WORK + '/dataset'
os.makedirs(DS, exist_ok=True)
have = [f for f in os.listdir(DS) if f.endswith('.wav')]
if have:
    print('دیتاست از پیش آماده است: %d تکه' % len(have))
else:
    segs, rep = D.buildDataset_(srcs, DS, sampleDir=OUT)
    for row in rep['files']:
        print('%-22s %7.1f ثانیه → %3d تکه'
              % (row['file'][:22], row['seconds'], row.get('segments', 0)))
    print('\n' + rep['line'])
    if not segs:
        raise SystemExit('هیچ تکه‌ای نماند — ضبط‌ها را ببینید')

## ۸ — آموزش

In [ ]:
P.preLog_(ROOT, VOICE)
env = P.env(ROOT)
steps = P.steps(VOICE, DS, ROOT, sr=SR, f0method='rmvpe', epochs=EPOCHS,
                save_every=SAVE_EVERY, version='v2', gpus='0', n_p=2,
                batch=BATCH, py=sys.executable, latest=1)
done = os.path.join(ROOT, 'logs', VOICE, '3_feature768')
for nm, cmd in steps:
    if nm in ('preprocess', 'extract_f0', 'extract_feature') \
            and os.path.isdir(done) and os.listdir(done):
        print('%s: از پیش انجام شده' % nm)
        continue
    if nm == 'train':
        info = P.preTrain_(ROOT, VOICE, sr=SR, version='v2')
        print('فهرستِ آموزش:', info)
        if not info['from_dataset']:
            raise SystemExit('فهرست خالی است — استخراج چیزی نساخت')
    t0 = time.time()
    print('\n=== %s ===' % nm, flush=True)
    r = subprocess.run(cmd, cwd=ROOT, env=env)
    print('%s: %ds' % (nm, time.time() - t0))
    if r.returncode:
        raise SystemExit('قدمِ «%s» شکست خورد (کد %d)' % (nm, r.returncode))

## ۹ — نتیجه

دو فایل در `/kaggle/working` می‌نشیند و در تبِ **Output** قابلِ دانلود
است. همین دو تا چیزی است که موتور به کار می‌برد.

پوشهٔ کار پاک می‌شود تا خروجی سبک بماند.

In [ ]:
o = P.outputs(VOICE, ROOT)
if not os.path.exists(o['model']):
    raise SystemExit('آموزش تمام شد ولی مدلی ساخته نشد: ' + o['model'])
saved = [shutil.copy(o['model'], OUT)]
for f in sorted(os.listdir(o['index_dir'])):
    if f.endswith('.index'):
        saved.append(shutil.copy(os.path.join(o['index_dir'], f), OUT))
for s in saved:
    print('%8.1f مگابایت  %s' % (os.path.getsize(s) / 1048576, s))

# چند گیگابایتِ میانی در خروجی جایی ندارد.
shutil.rmtree(WORK, ignore_errors=True)
shutil.rmtree(ROOT, ignore_errors=True)
print('\nتمام شد. از تبِ Output برشان دارید.')